# Word Embedding with Keras — Project Workflow

### Goal
Show, step by step, how raw English sentences are converted into numbers a neural network can use: **text -> word IDs -> equal-length sequences -> dense embedding vectors**. This notebook is the foundation for the IMDB sentiment project in `simplernn.ipynb`, where the same ideas are applied to 50,000 real reviews.

### Workflow

```
Sentences  ->  one_hot (word IDs)  ->  pad_sequences  ->  Embedding layer  ->  dense vectors (7, 8, 10)
```

| Stage | What happens | Notebook steps |
|---|---|---|
| 1. Prepare text | Create 7 sample sentences and set the vocabulary size (`voc_size = 10000`). | Steps 1-5 |
| 2. Words to IDs | `one_hot` hashes every word to an integer in `[1, 10000)`; the same word gets the same ID. | Step 6 |
| 3. Make lengths equal | `pad_sequences` pads (or truncates) every sentence to `sent_length = 8` with `0`s at the front. | Steps 7-9, 17 |
| 4. Build the embedding model | `Sequential` + `Embedding(10000, 10, input_length=8)` — a trainable lookup table of 100,000 values. | Steps 10-12 |
| 5. Get the vectors | `model.predict` replaces each word ID by its 10-number vector; output shape `(7, 8, 10)`. | Steps 13-15 |
| 6. Handle a single sentence | Use `reshape(1, -1)` to give the model a batch of one sentence, shape `(1, 8)`. | Step 16 |

### Key ideas to remember
- **IDs have no meaning by themselves** — they are only labels for words.
- **Padding is required** because neural networks need rectangular, fixed-shape input; `0` is reserved for padding.
- **Embedding = lookup table**: word ID `n` picks row `n` of a `(vocab_size, dim)` table. The values are random until the layer is trained, after which similar words get similar vectors.
- **Shapes**: input `(batch, sentence_length)` -> output `(batch, sentence_length, embedding_dim)`.

### How this connects to the rest of the project
`embedding.ipynb` (this notebook, concepts) -> `simplernn.ipynb` (train a Simple RNN on IMDB and save `simple_rnn_imdb.h5`) -> `prediction.ipynb` (predict new reviews) -> `main.py` (Streamlit app).

### Requirements
Python with `tensorflow` (2.15 in `requirements.txt`) and `numpy`.

### Step 1 — Import `one_hot` from Keras

**What this cell does:** imports the `one_hot` helper from `tensorflow.keras.preprocessing.text`.

**Why it is needed:** neural networks cannot read raw text, so every word must first be turned into a number. `one_hot` is the simplest tool for this: it takes a sentence and returns a list of integers, one per word.

**Important detail:** despite its name, `one_hot` does *not* return big 0/1 vectors. It uses a hashing trick: each word is hashed and mapped to an integer index between `1` and `vocab_size - 1`. Two different words can occasionally land on the same index (a *collision*), and the exact integers can differ between runs/environments.

In [32]:
from tensorflow.keras.preprocessing.text import one_hot

### Step 2 — Create the sample sentences (our tiny corpus)

**What this cell does:** defines a Python list `sent` that holds 7 short English sentences.

**Why it is needed:** we need some text to demonstrate how raw text becomes numbers and then dense vectors (embeddings). The sentences are intentionally small and share words (`the`, `of`, `glass`, `I am a good`) so that you can see the *same word getting the same index* in later cells.

**Note:** the sentences have different lengths (4, 4, 4, 5, 5, 5, 4 words). This difference is exactly why we need padding later.

In [33]:
### sentences
sent=[  'the glass of milk',
     'the glass of juice',
     'the cup of tea',
    'I am a good boy',
     'I am a good developer',
     'understand the meaning of words',
     'your videos are good',]

### Step 3 — Display the sentences

**What this cell does:** simply prints the variable `sent`.

**Why it is needed:** a quick sanity check to confirm that the list was created correctly before we start transforming it. In Jupyter, writing a variable name on the last line of a cell shows its value.

In [34]:
sent

['the glass of milk',
 'the glass of juice',
 'the cup of tea',
 'I am a good boy',
 'I am a good developer',
 'understand the meaning of words',
 'your videos are good']

### Step 4 — Define the vocabulary size

**What this cell does:** sets `voc_size = 10000`.

**Why it is needed:** this number is the size of our *vocabulary space*. It plays two roles later:
1. `one_hot` maps each word to an integer in the range `[1, 10000)`.
2. The Embedding layer creates one row (vector) for each of these 10,000 possible indices.

**Rule of thumb:** a larger vocabulary size means fewer hash collisions but a bigger embedding table (more parameters and memory).

In [35]:
## Define the vocabulary size
voc_size=10000

### Step 5 — Display the sentences again

**What this cell does:** prints `sent` once more.

**Why it is needed:** it is only a quick re-check of the input data right before we convert it. It does not change anything and can safely be removed.

In [36]:
sent

['the glass of milk',
 'the glass of juice',
 'the cup of tea',
 'I am a good boy',
 'I am a good developer',
 'understand the meaning of words',
 'your videos are good']

### Step 6 — One-hot (index) representation of each sentence

**The code**

```python
one_hot_repr=[one_hot(words,voc_size)for words in sent]
one_hot_repr
```

**1. The list comprehension**

`[one_hot(words, voc_size) for words in sent]` is shorthand for:

```python
one_hot_repr = []
for words in sent:                      # each sentence, e.g. 'the glass of milk'
    one_hot_repr.append(one_hot(words, voc_size))
```

- `sent` is the list of 7 sentences (strings).
- `words` is a loop variable that holds **one whole sentence** at a time, not a single word (`sentence` would be a clearer name).
- The result is a list of 7 lists, one per sentence.

**2. What `one_hot(text, n)` does**

Here `text` is the sentence (`words`) and `n` is the vocabulary size (`voc_size = 10000`). Internally it:
1. Lower-cases the sentence and strips punctuation.
2. Splits it on spaces, e.g. `'the glass of milk'` becomes `['the', 'glass', 'of', 'milk']`.
3. Hashes each word to an integer between `1` and `n - 1`. The same word always gives the same integer within a run.

Despite its name, it does **not** return 0/1 vectors. It returns word IDs (a hashing-trick shortcut). A true one-hot vector would be 10,000 long with a single `1`.

**3. Reading the output**

```
[[6186, 6775, 637, 4895],        # 'the glass of milk'
 [6186, 6775, 637, 5462],        # 'the glass of juice'
 [6186, 7872, 637, 7943],        # 'the cup of tea'
 [7793, 3440, 840, 9701, 4879],  # 'I am a good boy'
 [7793, 3440, 840, 9701, 1294],  # 'I am a good developer'
 [932, 6186, 733, 637, 1731],    # 'understand the meaning of words'
 [2258, 439, 5397, 9701]]        # 'your videos are good'
```

- `the` is always `6186` and `of` is always `637`.
- `glass` is `6775` in sentences 1 and 2, and `good` is `9701` in sentences 4, 5 and 7.
- Sentences 1 and 2 differ only in the last word (`milk` = `4895`, `juice` = `5462`).

**4. The last line, `one_hot_repr`**

A bare variable name on the last line of a Jupyter cell prints its value, which is why the list of lists appears.

**5. Caveats**

- **Collisions:** the hash can map two different words to the same ID. This is a real risk when `voc_size` is small compared to the real vocabulary.
- **Not stable across runs:** the exact numbers can change between sessions or TensorFlow versions, so do not hard-code them.
- **Different lengths:** the lists have 4 or 5 numbers each, which is why the next steps use `pad_sequences` to make them all length 8.
- **No meaning yet:** the IDs are arbitrary labels. Similar words only become close together after passing through an `Embedding` layer and being trained.

**Next step:** these lists go into `pad_sequences` (fixed length), and the padded matrix then goes into the `Embedding` layer, which turns each ID into a 10-number vector.

In [37]:
### One Hot Representation
one_hot_repr=[one_hot(words,voc_size)for words in sent]
one_hot_repr

[[4046, 4598, 672, 5728],
 [4046, 4598, 672, 7557],
 [4046, 4832, 672, 1998],
 [6381, 5129, 7556, 4044, 5691],
 [6381, 5129, 7556, 4044, 8844],
 [3744, 4046, 4874, 672, 9658],
 [1889, 1147, 8865, 4044]]

### Step 7 — Import the layers and utilities for the embedding model

**What this cell does:** imports three things:
- `Embedding` — the layer that converts integer word IDs into dense vectors.
- `pad_sequences` — makes all sequences the same length (the commented-out line is an old import path that no longer works in newer TensorFlow versions).
- `Sequential` — a simple container to stack layers one after another.

**Why it is needed:** these are the building blocks for the padding step and the embedding model that follow.

In [38]:
## word Embedding Representation

from tensorflow.keras.layers import Embedding
#from tensorflow.keras.processing.sequence import pad_sequences
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.models import Sequential

### Step 8 — Import NumPy

**What this cell does:** imports `numpy` as `np`.

**Why it is needed:** NumPy is the standard array library used by TensorFlow/Keras. `pad_sequences` returns a NumPy array and the model's predictions are NumPy arrays too. (It is not directly called in this notebook, but it is a standard import for working with these arrays.)

In [39]:
import numpy as np

### Step 9 — Pad all sentences to the same length

**What this cell does:**
- Sets `sent_length = 8`, the fixed length we want for every sentence.
- Calls `pad_sequences(one_hot_repr, padding='pre', maxlen=8)` to make every sentence exactly 8 numbers long.

**How padding works:**
- `padding='pre'` adds the filler value `0` at the **beginning** of shorter sentences (e.g. `[6186, 6775, 637, 4895]` becomes `[0, 0, 0, 0, 6186, 6775, 637, 4895]`).
- Sentences longer than `maxlen` would be truncated (none are here).

**Why it is needed:** neural networks process data in batches/matrices, so every input must have the same shape. The result is a `(7, 8)` matrix: 7 sentences × 8 positions. Index `0` is reserved for padding, which is why `one_hot` starts real words at `1`.

In [40]:
sent_length=8
embedded_docs=pad_sequences(one_hot_repr,padding='pre',maxlen=sent_length)
print(embedded_docs)

[[   0    0    0    0 4046 4598  672 5728]
 [   0    0    0    0 4046 4598  672 7557]
 [   0    0    0    0 4046 4832  672 1998]
 [   0    0    0 6381 5129 7556 4044 5691]
 [   0    0    0 6381 5129 7556 4044 8844]
 [   0    0    0 3744 4046 4874  672 9658]
 [   0    0    0    0 1889 1147 8865 4044]]


### Step 10 — Choose the embedding dimension

**What this cell does:** sets `dim = 10`.

**Why it is needed:** this is the length of the vector that will represent each word. Instead of a word being one integer (or a 10,000-long one-hot vector), it becomes a dense vector of 10 learnable numbers.

**Trade-off:** a bigger dimension can capture richer meaning but needs more data and parameters. Real-world models often use 50–300 or more; 10 is chosen here only to keep the output readable.

In [41]:
## feature representation
dim=10

### Step 11 — Build and compile the embedding model

**What this cell does:**
1. `Sequential()` creates an empty model.
2. `Embedding(voc_size, dim, input_length=sent_length)` adds the embedding layer:
   - `voc_size = 10000` → number of rows in the lookup table,
   - `dim = 10` → size of each word vector,
   - `input_length = 8` → each input sentence has 8 word IDs.
3. `model.compile('adam', 'mse')` chooses the optimizer (Adam) and loss (mean squared error).

**Why compile if we are not training?** Keras needs a compiled model to be fully built before we call `predict`. The optimizer/loss values are not actually used here — we only want to *look up* vectors.

**Concept:** an Embedding layer is essentially a trainable lookup table of shape `(10000, 10)`. Word ID `n` simply picks row `n`. The values start random and would be learned during real training.

In [42]:
model=Sequential()
model.add(Embedding(voc_size,dim,input_length=sent_length))
model.compile('adam','mse')

### Step 12 — Inspect the model architecture

**What this cell does:** prints the model summary (layers, output shapes, parameter counts).

**How to read the output:**
- Output shape `(None, 8, 10)` → `None` is the batch size (any number of sentences), `8` words per sentence, `10` numbers per word.
- Parameters = `10,000 × 10 = 100,000`, i.e. one 10-number vector for each of the 10,000 possible word IDs.

All 100,000 parameters are *trainable*, meaning they would be adjusted when training on a real task.

In [43]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Step 13 — Get embeddings for all padded sentences

**What this cell does:** passes the whole `embedded_docs` matrix `(7, 8)` through the model with `model.predict(...)`.

**Output:** a 3-D array of shape `(7, 8, 10)` — for each of the 7 sentences, each of the 8 positions is replaced by its 10-dimensional vector.

**What to observe:**
- All the padding positions (index `0`) have exactly the same vector, because they all look up row 0 of the table.
- The same word (e.g. `the`) gets an identical vector in different sentences.
- The numbers are small random values since the layer has **not been trained** — they only become meaningful (similar words → similar vectors) after training on a task.

In [44]:
model.predict(embedded_docs)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step


array([[[-3.65978107e-02, -3.89938951e-02,  8.00769404e-03,
         -1.00893006e-02, -4.31101397e-03, -3.22313085e-02,
          1.60870887e-02,  4.66421992e-03, -2.87811998e-02,
         -2.39587948e-03],
        [-3.65978107e-02, -3.89938951e-02,  8.00769404e-03,
         -1.00893006e-02, -4.31101397e-03, -3.22313085e-02,
          1.60870887e-02,  4.66421992e-03, -2.87811998e-02,
         -2.39587948e-03],
        [-3.65978107e-02, -3.89938951e-02,  8.00769404e-03,
         -1.00893006e-02, -4.31101397e-03, -3.22313085e-02,
          1.60870887e-02,  4.66421992e-03, -2.87811998e-02,
         -2.39587948e-03],
        [-3.65978107e-02, -3.89938951e-02,  8.00769404e-03,
         -1.00893006e-02, -4.31101397e-03, -3.22313085e-02,
          1.60870887e-02,  4.66421992e-03, -2.87811998e-02,
         -2.39587948e-03],
        [-3.84594128e-03,  3.93045805e-02, -3.55757475e-02,
         -4.87945788e-02,  3.55067514e-02,  3.71210836e-02,
         -2.16975342e-02, -4.46796790e-02, -2.095205

### Step 14 — Look at the first padded sentence

**What this cell does:** displays `embedded_docs[0]`, the first row of the padded matrix.

**Why it is needed:** it shows the exact input for sentence 1 (`'the glass of milk'`): `[0 0 0 0 6186 6775 637 4895]`. This lets us compare the integer input with the vectors produced in the next cell.

In [45]:
embedded_docs[0]

array([   0,    0,    0,    0, 4046, 4598,  672, 5728], dtype=int32)

In [48]:
embedded_docs[0].shape

(8,)

### Step 15 — Get the embedding for a single sentence's IDs

**What this cell does:** runs `model.predict(embedded_docs[0])` on the first row only.

**Important detail:** `embedded_docs[0]` is a 1-D array of 8 integers. Keras treats each integer as a *separate sample* of length 1, so the output has shape `(8, 10)` — one 10-number vector per word ID (the first four rows, for padding, are identical).

**Tip:** to predict a *single sentence* in the same shape as the training input, you would pass a 2-D array, e.g. `embedded_docs[0].reshape(1, -1)` or `embedded_docs[:1]`, which would return shape `(1, 8, 10)`.

The error occurs because Keras models expect the input data to have a batch dimension. embedded_docs[0] is a 1D array of shape (8,), so Keras cannot interpret its shape correctly during inference.

Using embedded_docs[0].reshape(1, -1) is an excellent way to solve this. It converts the 1D array of shape (8,) into a 2D array of shape (1, 8), which adds the batch dimension that model.predict() expects.

In [49]:
model.predict(embedded_docs[0].reshape(1, -1))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step


array([[[-0.03659781, -0.0389939 ,  0.00800769, -0.0100893 ,
         -0.00431101, -0.03223131,  0.01608709,  0.00466422,
         -0.0287812 , -0.00239588],
        [-0.03659781, -0.0389939 ,  0.00800769, -0.0100893 ,
         -0.00431101, -0.03223131,  0.01608709,  0.00466422,
         -0.0287812 , -0.00239588],
        [-0.03659781, -0.0389939 ,  0.00800769, -0.0100893 ,
         -0.00431101, -0.03223131,  0.01608709,  0.00466422,
         -0.0287812 , -0.00239588],
        [-0.03659781, -0.0389939 ,  0.00800769, -0.0100893 ,
         -0.00431101, -0.03223131,  0.01608709,  0.00466422,
         -0.0287812 , -0.00239588],
        [-0.00384594,  0.03930458, -0.03557575, -0.04879458,
          0.03550675,  0.03712108, -0.02169753, -0.04467968,
         -0.02095206, -0.02173244],
        [ 0.01571235,  0.00237502, -0.04186605,  0.01626179,
          0.01606126,  0.02499259,  0.02395718, -0.02113512,
          0.01102019, -0.00851647],
        [ 0.04636114,  0.00468135, -0.02582498,  0.0

In [50]:
embedded_docs[0].shape

(8,)

extra knowledge


### Step 16 — Predict a single sentence correctly using `reshape`

**What `reshape` does:** it changes how an array's elements are arranged into dimensions. It never changes the elements or their order.

**Why we need it here**

```python
embedded_docs[0]                 # shape (8,)   -> [0 0 0 0 6186 6775 637 4895]
embedded_docs[0].reshape(1, -1)  # shape (1, 8) -> [[0 0 0 0 6186 6775 637 4895]]
```

- `embedded_docs[0]` is a 1-D array with 8 numbers.
- `reshape(1, -1)` makes it 2-D: 1 row and 8 columns.
- The `Embedding` layer expects `(batch_size, sent_length)`. A shape of `(1, 8)` means "a batch of 1 sentence with 8 word IDs".
- With the plain `(8,)` array (Step 15), Keras treats each number as its own one-word sample, so the output was `(8, 10)`.
- With `(1, 8)` the output is `(1, 8, 10)`: one sentence, 8 words, 10 numbers per word.

**The rules of `reshape`**

1. **Total elements must stay the same.** The product of the new shape must equal the old size. 8 elements can become `(1, 8)`, `(2, 4)`, `(4, 2)` or `(8, 1)`, but not `(3, 3)`.
2. **`-1` means "work this dimension out for me".** NumPy computes it from the other dimensions. For 8 elements, `reshape(1, -1)` gives `(1, 8)` and `reshape(-1, 2)` gives `(4, 2)`. Only one `-1` is allowed per call.
3. **Order is preserved.** Elements are read row by row, left to right, and refilled into the new shape.

**Examples**

```python
a = np.array([1, 2, 3, 4, 5, 6])      # shape (6,)

a.reshape(2, 3)    # [[1 2 3]
                   #  [4 5 6]]       shape (2, 3)
a.reshape(3, 2)    # [[1 2]
                   #  [3 4]
                   #  [5 6]]         shape (3, 2)
a.reshape(-1, 2)   # same as (3, 2), NumPy works out the 3
a.reshape(1, -1)   # [[1 2 3 4 5 6]] shape (1, 6)
a.reshape(-1, 1)   # a column: [[1] [2] [3] [4] [5] [6]]  shape (6, 1)
a.reshape(4, 2)    # ValueError: cannot reshape array of size 6 into shape (4,2)
```

**Equivalent ways to add the batch dimension**

```python
embedded_docs[0].reshape(1, -1)    # shape (1, 8)
embedded_docs[:1]                  # shape (1, 8), slicing keeps the row dimension
embedded_docs[0][np.newaxis, :]    # shape (1, 8)
np.expand_dims(embedded_docs[0], 0)
```

`reshape` returns a view of the same data when it can, not a copy, so changing the reshaped array can change the original.

**What the code cell below does:** it reshapes the first sentence to `(1, 8)` and predicts, so the output has shape `(1, 8, 10)` (one sentence, 8 words, 10-number vectors). It then shows two of the `reshape` examples above. Run it to see the shapes.

### Step 17 — Why is padding required?

**The problem:** our sentences have different lengths (4 or 5 word IDs), but a neural network works on **batches stored as one rectangular matrix** (a tensor). Every row must have the same number of columns, so a list of unequal lists cannot be turned into a proper `(num_sentences, sent_length)` array.

```
[6186, 6775, 637, 4895]          # 4 numbers
[7793, 3440, 840, 9701, 4879]    # 5 numbers  -> cannot be stacked into one matrix
```

**Reasons padding is required**

1. **Fixed input shape.** The `Embedding` layer was built with `input_length=8`, and later layers (e.g. `SimpleRNN`/`LSTM`/`Dense`) also expect a fixed shape. Ragged input raises an error.
2. **Batch processing.** Training sends many sentences at once (e.g. 32) for speed on CPU/GPU. This only works when all sentences in the batch have the same length.
3. **Efficient vectorised math.** Matrix operations are fast because every row has identical size. Unequal rows would force slow Python loops.
4. **A consistent length for new data.** Any new sentence at prediction time must be padded/truncated to the same length used in training (e.g. 8 here, 500 in the IMDB project).

**How `pad_sequences` fixes it**

- It fills short sequences with the value `0` until they reach `maxlen`. This is why real word IDs start from `1` (`0` is reserved for padding).
- `padding='pre'` adds the zeros at the **beginning** (`[0 0 0 0 6186 6775 637 4895]`); `padding='post'` adds them at the **end** (`[6186 6775 637 4895 0 0 0 0]`).
- Sequences **longer** than `maxlen` are cut. By default `truncating='pre'` removes numbers from the start; use `truncating='post'` to remove from the end.

**Which side should we pad?** For RNN/LSTM models, `'pre'` is usually preferred because the real words sit right next to the final output step, so the network does not have to carry information across many padding steps.

**Choosing `maxlen`:** too small loses information from long sentences; too large wastes memory and time on zeros. A common choice is a length that covers most (e.g. 95%) of the sentences.

**Side note:** the model treats padding `0`s as real input unless told otherwise. Use `Embedding(..., mask_zero=True)` so that later layers ignore the padding positions.

**What the code cell below does:** it first tries to build an array from the unequal lists (this fails or creates an object array), then shows `pad_sequences` with pre-padding, post-padding and truncation. Run it to see the results.

In [51]:
ragged = [[6186, 6775, 637, 4895],
          [7793, 3440, 840, 9701, 4879]]

# 1) Without padding: unequal lengths cannot form a normal matrix
try:
    print(np.array(ragged).shape)
except ValueError as e:
    print("Error without padding:", e)

# 2) Pre-padding (zeros at the start)
print(pad_sequences(ragged, padding='pre', maxlen=8))

# 3) Post-padding (zeros at the end)
print(pad_sequences(ragged, padding='post', maxlen=8))

# 4) Truncation: maxlen shorter than the sentence
print(pad_sequences(ragged, maxlen=3))                       # cuts from the start
print(pad_sequences(ragged, maxlen=3, truncating='post'))    # cuts from the end

Error without padding: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.
[[   0    0    0    0 6186 6775  637 4895]
 [   0    0    0 7793 3440  840 9701 4879]]
[[6186 6775  637 4895    0    0    0    0]
 [7793 3440  840 9701 4879    0    0    0]]
[[6775  637 4895]
 [ 840 9701 4879]]
[[6186 6775  637]
 [7793 3440  840]]
